In [37]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from tabulate import tabulate

movies = pd.read_csv(r"movies.csv")   # from MovieLens dataset
ratings = pd.read_csv(r"ratings.csv")

ratings = ratings[ratings['rating'] >= 4.0]

data = pd.merge(ratings, movies, on='movieId')

popular_movies = data['title'].value_counts().head(100).index
data_small = data[data['title'].isin(popular_movies)]
data_small = data_small[data_small['userId'] <= 1000]

basket = (data_small.groupby(['userId', 'title'])['rating']
          .count().unstack().reset_index().fillna(0)
          .set_index('userId'))

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

frequent_itemsets = fpgrowth(basket, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

def format_movies(movie_set, max_movies=3):
    movie_list = list(movie_set)
    if len(movie_list) > max_movies:
        return ', '.join(movie_list[:max_movies]) + ', ...'
    else:
        return ', '.join(movie_list)

print("\n--- Frequent Itemsets (Top 10) ---")
fi_top10 = frequent_itemsets.sort_values(by='support', ascending=False).head(10).copy()
fi_top10['Movies'] = fi_top10['itemsets'].apply(lambda x: format_movies(x, max_movies=3))
print(tabulate(fi_top10[['Movies','support']], headers=['Movie(s)','Support'], 
               showindex=True, tablefmt='fancy_grid', floatfmt=".3f"))

rules_top10 = rules.sort_values(by='confidence', ascending=False).head(10).copy()
rules_top10['LHS (If liked)'] = rules_top10['antecedents'].apply(lambda x: format_movies(x))
rules_top10['RHS (Then also liked)'] = rules_top10['consequents'].apply(lambda x: format_movies(x))

print("\n--- Association Rules (Top 10) ---")
print(tabulate(rules_top10[['LHS (If liked)','RHS (Then also liked)','support','confidence','lift']],
               headers=['LHS (If liked)','RHS (Then also liked)','Support','Confidence','Lift'],
               showindex=False, tablefmt='fancy_grid', floatfmt=".3f"))

target_movie = "Inception (2010)"
recommendations = rules[rules['antecedents'] == frozenset([target_movie])].copy()
recommendations['consequents'] = recommendations['consequents'].apply(lambda x: format_movies(x))

print(f"\n--- Recommendations for users who liked '{target_movie}' ---")
rec_display = recommendations[['consequents','confidence','lift']].head(10)
print(tabulate(rec_display, headers=['Suggested Movie(s)','Confidence','Lift'], 
               showindex=False, tablefmt='fancy_grid', floatfmt=".2f"))


C:\Users\jugal\AppData\Local\Temp\ipykernel_19712\1571930901.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)
D:\anaconda\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(



--- Frequent Itemsets (Top 10) ---
╒═════╤═══════════════════════════════════════════════════════╤═══════════╕
│     │ Movie(s)                                              │   Support │
╞═════╪═══════════════════════════════════════════════════════╪═══════════╡
│  54 │ Shawshank Redemption, The (1994)                      │     0.467 │
├─────┼───────────────────────────────────────────────────────┼───────────┤
│   0 │ Forrest Gump (1994)                                   │     0.424 │
├─────┼───────────────────────────────────────────────────────┼───────────┤
│  51 │ Pulp Fiction (1994)                                   │     0.416 │
├─────┼───────────────────────────────────────────────────────┼───────────┤
│   1 │ Silence of the Lambs, The (1991)                      │     0.383 │
├─────┼───────────────────────────────────────────────────────┼───────────┤
│   2 │ Matrix, The (1999)                                    │     0.378 │
├─────┼─────────────────────────────────────────────